# 03 — Building Characteristics (Paris / EUBUCCO)

Aggregates EUBUCCO building-level statistics per grid cell.

**Replaces:** NYC PLUTO notebook 03 with EUBUCCO open data.

**Features extracted per cell:**
- `avg_height` — mean building height (m)
- `avg_floors` — mean number of floors
- `avg_construction_year` — mean year built
- `building_count` — number of buildings in cell
- `residential_ratio` — fraction of residential buildings

**Output file:** `csv/Paris/03_building_characteristics.csv`

In [ ]:
PARIS_CONFIG = "paris.json"

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import os
import math

with open(PARIS_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

NUTS_CODE   = config["nuts_code"]
CELL_SIZE_M = config["grid_cell_size_m"]
CSV_DIR     = config["csv_dir"]
os.makedirs(CSV_DIR, exist_ok=True)

# Load grid
df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
valid_cells = set(df_grid["cell_id"])
print(f"Loaded {len(df_grid)} grid cells")

# Recover grid parameters saved by notebook 01
gp      = config["grid_params"]
LAT_MIN = gp["lat_min"]
LON_MIN = gp["lon_min"]
LAT_STEP = gp["lat_step"]
LON_STEP = gp["lon_step"]
print(f"Grid params loaded from paris.json")

In [ ]:
# ── Stream EUBUCCO ────────────────────────────────────
storage_opts = {
    "anon": True,
    "client_kwargs": {"endpoint_url": "https://s3.eubucco.com"}
}
path = f"s3://eubucco/v0.2/buildings/parquet/nuts_id={NUTS_CODE}/{NUTS_CODE}.parquet"
print(f"Streaming EUBUCCO...")

gdf = gpd.read_parquet(path, storage_options=storage_opts)
gdf = gdf.to_crs("EPSG:4326")
gdf["latitude"]  = gdf.geometry.centroid.y
gdf["longitude"] = gdf.geometry.centroid.x
gdf = gdf.dropna(subset=["latitude", "longitude"])
print(f"Loaded {len(gdf):,} buildings")

In [ ]:
# ── Assign buildings to grid cells ────────────────────
gdf["grid_row"] = ((gdf["latitude"]  - LAT_MIN) / LAT_STEP).astype(int)
gdf["grid_col"] = ((gdf["longitude"] - LON_MIN) / LON_STEP).astype(int)
gdf["cell_id"]  = "r" + gdf["grid_row"].astype(str).str.zfill(4) + "_c" + gdf["grid_col"].astype(str).str.zfill(4)

# Keep only buildings in valid cells
gdf = gdf[gdf["cell_id"].isin(valid_cells)].copy()
print(f"Buildings in valid grid cells: {len(gdf):,}")

# Clean numeric fields
for col in ["height", "floors", "construction_year"]:
    gdf[col] = pd.to_numeric(gdf[col], errors="coerce")

gdf.loc[gdf["construction_year"] < 1000, "construction_year"] = np.nan
gdf.loc[gdf["height"] <= 0, "height"] = np.nan
gdf.loc[gdf["floors"] <= 0, "floors"] = np.nan

In [ ]:
# ── Aggregate per grid cell ───────────────────────────
gdf["is_residential"] = (gdf["type"] == "residential").astype(int)

agg = gdf.groupby("cell_id").agg(
    avg_height            = ("height",            "mean"),
    avg_floors            = ("floors",            "mean"),
    avg_construction_year = ("construction_year", "mean"),
    building_count        = ("cell_id",           "count"),
    residential_ratio     = ("is_residential",    "mean"),
).reset_index()

agg["avg_height"]            = agg["avg_height"].round(1)
agg["avg_floors"]            = agg["avg_floors"].round(1)
agg["avg_construction_year"] = agg["avg_construction_year"].round(0).astype("Int64")
agg["residential_ratio"]     = agg["residential_ratio"].round(3)

# Ensure all grid cells are present
df_result = df_grid[["cell_id"]].merge(agg, on="cell_id", how="left")
print(f"Aggregated {len(df_result)} cells")
print(df_result.describe().round(2).to_string())

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/03_building_characteristics.csv"
df_result.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_result)} rows x {df_result.shape[1]} cols)")
df_result.head(10)